In [1]:
%pip install --upgrade pip
%pip install --upgrade ucimlrepo matplotlib xarray

import sys; sys.path.append('../source')
import importlib, mushroom; importlib.reload(mushroom)
from ucimlrepo import fetch_ucirepo
import pandas

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
pandas.set_option('display.max_rows', None)

# fetch dataset
dataset = fetch_ucirepo(id=73)
analyzer = mushroom.Analyzer(dataset)

print('features')
print('********************************************')
print(dataset.metadata['additional_info']['variable_info'])

print()

print('feature spaces (all ' + str(len(analyzer.feature_spaces)) + ' possible combinations)')
print('********************************************')
print('\n'.join(f'{i+1}. {feature_space[0]}, {feature_space[1]}' for i, feature_space in enumerate(analyzer.feature_spaces)))

features
********************************************
     1. cap-shape:                bell=b,conical=c,convex=x,flat=f, knobbed=k,sunken=s
     2. cap-surface:              fibrous=f,grooves=g,scaly=y,smooth=s
     3. cap-color:                brown=n,buff=b,cinnamon=c,gray=g,green=r, pink=p,purple=u,red=e,white=w,yellow=y
     4. bruises?:                 bruises=t,no=f
     5. odor:                     almond=a,anise=l,creosote=c,fishy=y,foul=f, musty=m,none=n,pungent=p,spicy=s
     6. gill-attachment:          attached=a,descending=d,free=f,notched=n
     7. gill-spacing:             close=c,crowded=w,distant=d
     8. gill-size:                broad=b,narrow=n
     9. gill-color:               black=k,brown=n,buff=b,chocolate=h,gray=g, green=r,orange=o,pink=p,purple=u,red=e, white=w,yellow=y
    10. stalk-shape:              enlarging=e,tapering=t
    11. stalk-root:               bulbous=b,club=c,cup=u,equal=e, rhizomorphs=z,rooted=r,missing=?
    12. stalk-surface-above-ring: f

In [3]:
edible_only = analyzer.find_minimal_edibility_signals('e')
poisonous_only = analyzer.find_minimal_edibility_signals('p')

print('Edible-only signals (minimal set)')
print('**************************************************************')
print('number of edible mushrooms expected to account for', analyzer.edibility_count['e'])
print('number of edible mushrooms actually accounted for', edible_only['edible_count'].sum())
display(edible_only)

print()

print('Poisonous-only signals (minimal set)')
print('**************************************************************')
print('number of poisonous mushrooms expected to account for', analyzer.edibility_count['p'])
print('number of poisonous mushrooms actually accounted for', poisonous_only['poisonous_count'].sum())
display(poisonous_only)

Edible-only signals (minimal set)
**************************************************************
number of edible mushrooms expected to account for 4208
number of edible mushrooms actually accounted for 4208


,feature1_name,feature1_value,feature2_name,feature2_value,edible_count,poisonous_count
0,odor,n,stalk-shape,t,2496,0
1,ring-number,t,spore-print-color,w,528,0
2,stalk-root,c,ring-type,p,512,0
3,gill-attachment,a,ring-type,p,192,0
4,stalk-root,r,stalk-surface-above-ring,s,192,0
5,cap-surface,f,stalk-root,e,96,0
6,gill-spacing,w,stalk-shape,t,96,0
7,bruises,f,stalk-surface-below-ring,f,72,0
8,stalk-surface-below-ring,s,stalk-color-below-ring,n,24,0



Poisonous-only signals (minimal set)
**************************************************************
number of poisonous mushrooms expected to account for 3916
number of poisonous mushrooms actually accounted for 3916


,feature1_name,feature1_value,feature2_name,feature2_value,edible_count,poisonous_count
0,gill-spacing,c,stalk-surface-above-ring,k,0,2228
1,gill-color,b,ring-number,o,0,864
2,stalk-shape,t,spore-print-color,h,0,288
3,odor,p,stalk-color-below-ring,w,0,256
4,odor,c,veil-color,w,0,192
5,bruises,t,spore-print-color,r,0,72
6,gill-size,n,population,c,0,16


In [4]:
# Verify: Can the edible-only set (or poisonous-only set) classify every mushroom?
# Logic: match any edibility signal in the set -> that class; no match -> the opposite class.

data = dataset.data.original

def matches_rule(row, rules):
    """Return True if the mushroom matches at least one edibility signal in the set."""
    for _, rule in rules.iterrows():
        if (row[rule['feature1_name']] == rule['feature1_value']
                and row[rule['feature2_name']] == rule['feature2_value']):
            return True
    return False

# --- Edible-only classifier: match -> edible, no match -> poisonous ---
correct = 0
for _, row in data.iterrows():
    predicted = 'e' if matches_rule(row, edible_only) else 'p'
    if predicted == row['poisonous']:
        correct += 1

print('Edible-only set (match -> edible, no match -> poisonous)')
print('-' * 60)
print(f'Rules: {len(edible_only)}')
print(f'Correct: {correct} / {len(data)} ({correct / len(data) * 100:.1f}%)')

# --- Poisonous-only classifier: match -> poisonous, no match -> edible ---
correct = 0
for _, row in data.iterrows():
    predicted = 'p' if matches_rule(row, poisonous_only) else 'e'
    if predicted == row['poisonous']:
        correct += 1

print()
print('Poisonous-only set (match -> poisonous, no match -> edible)')
print('-' * 60)
print(f'Rules: {len(poisonous_only)}')
print(f'Correct: {correct} / {len(data)} ({correct / len(data) * 100:.1f}%)')

# --- Conclusion ---
print()
print('Hypothesis confirmed: either minimal set alone determines edibility for every mushroom.')
print(f'The poisonous-only set is more efficient ({len(poisonous_only)} rules vs {len(edible_only)}).')

Edible-only set (match -> edible, no match -> poisonous)
------------------------------------------------------------
Rules: 9
Correct: 8124 / 8124 (100.0%)

Poisonous-only set (match -> poisonous, no match -> edible)
------------------------------------------------------------
Rules: 7
Correct: 8124 / 8124 (100.0%)

Hypothesis confirmed: either minimal set alone determines edibility for every mushroom.
The poisonous-only set is more efficient (7 rules vs 9).
